# fullstack — local Spark (browser)

This notebook runs the **local pipeline**: (**1**) PySpark customers logic writing Parquet, (**2**) Scala **`TokenizeEmailsJob`** via `spark-submit` and [`run-local.sh`](../scala/tokenize-email-job/run-local.sh). You don’t need a separate Scala notebook for that step—a **`%%bash`** cell invokes the shell the same way as your terminal ([`TokenizeEmailsJob.scala`](../scala/tokenize-email-job/src/main/scala/fullstack/jobs/TokenizeEmailsJob.scala)).

Paths resolve from **repository root** (walking up to `pyproject.toml`).

**Run in the browser:** from the repo root, with Jupyter installed:

```bash
poetry install --extras notebook
poetry run jupyter notebook notebooks/fullstack_local_spark.ipynb
```

Or JupyterLab: `poetry run jupyter lab notebooks/`

If **`import fullstack`** fails, start Jupyter with **`poetry run jupyter …`** or register the env: **`poetry run python -m ipykernel install --user --name fullstack`** and pick kernel **fullstack**.

Note: **`fullstack.jobs.customers_job.run()`** calls **`spark.stop()`** locally. This notebook avoids stopping early so PySpark stays interactive; uncomment **`spark.stop()`** when finished.


In [8]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file():
            return path
    raise RuntimeError(
        "Could not find pyproject.toml. cd to the fullstack repo or open this notebook from that tree."
    )


REPO_ROOT = find_repo_root(Path.cwd())
src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

OUTPUT_PARQUET = REPO_ROOT / "src/fullstack/jobs/output"
OUTPUT_PARQUET.mkdir(parents=True, exist_ok=True)
TOKENIZED_PARQUET = REPO_ROOT / "src/fullstack/jobs/output_tokenized"

print("REPO_ROOT:", REPO_ROOT)
print("Customers Parquet (Python job → Scala input):", OUTPUT_PARQUET)
print("Tokenized Parquet (Scala job output):", TOKENIZED_PARQUET)

REPO_ROOT: /Users/markcosciello/git/fullstack
Customers Parquet (Python job → Scala input): /Users/markcosciello/git/fullstack/src/fullstack/jobs/output
Tokenized Parquet (Scala job output): /Users/markcosciello/git/fullstack/src/fullstack/jobs/output_tokenized


In [9]:
from pyspark.sql import functions as F

from fullstack.spark_session import get_spark, is_databricks_runtime

spark = get_spark(app_name="fullstack-notebook")
print("Databricks runtime:", is_databricks_runtime())
print("Spark:", spark.version)

Databricks runtime: False
Spark: 3.5.8


In [10]:
customers_df = spark.createDataFrame(
    [
        (1, "Alice", "alice@example.com"),
        (2, "Bob", "bob@example.com"),
        (3, "Carlos", "carlos@example.com"),
    ],
    ["customer_id", "name", "email"],
)

transformed_df = (
    customers_df.withColumn("ingested_at", F.to_timestamp(F.lit("2026-01-01 09:30:00")))
    .repartition(2)
)

transformed_df.show(truncate=False)
transformed_df.printSchema()

[Stage 0:>                                                          (0 + 8) / 8]

+-----------+------+------------------+-------------------+
|customer_id|name  |email             |ingested_at        |
+-----------+------+------------------+-------------------+
|1          |Alice |alice@example.com |2026-01-01 09:30:00|
|2          |Bob   |bob@example.com   |2026-01-01 09:30:00|
|3          |Carlos|carlos@example.com|2026-01-01 09:30:00|
+-----------+------+------------------+-------------------+

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [11]:
transformed_df.write.mode("overwrite").parquet(str(OUTPUT_PARQUET))
print("Wrote:", OUTPUT_PARQUET)

Wrote: /Users/markcosciello/git/fullstack/src/fullstack/jobs/output


In [12]:
read_back = spark.read.parquet(str(OUTPUT_PARQUET))
read_back.orderBy("customer_id").show(truncate=False)

+-----------+------+------------------+-------------------+
|customer_id|name  |email             |ingested_at        |
+-----------+------+------------------+-------------------+
|1          |Alice |alice@example.com |2026-01-01 09:30:00|
|2          |Bob   |bob@example.com   |2026-01-01 09:30:00|
|3          |Carlos|carlos@example.com|2026-01-01 09:30:00|
+-----------+------+------------------+-------------------+



## Scala: `TokenizeEmailsJob` + `run-local.sh`

The Scala job ([`TokenizeEmailsJob.scala`](../scala/tokenize-email-job/src/main/scala/fullstack/jobs/TokenizeEmailsJob.scala)) reads the customers Parquet directory, adds token columns, and writes to **`output_tokenized`**. Launcher: [`run-local.sh`](../scala/tokenize-email-job/run-local.sh) (Poetry **`spark-submit`**, **`local[*]`**).

**Magic cells:** **`%%bash`** below is ordinary shell—the same commands you would paste in zsh/bash. (**`%%sh`** behaves the same in IPython.)

**Separate Scala notebook?** Not required here. Submitting the packaged JAR from this Python notebook matches your repo scripts. Use a Scala kernel notebook only if you want interactive Scala snippets against an attached **`spark`** without `spark-submit`.

**Note:** PySpark above and **`spark-submit`** are different processes. Optionally run **`spark.stop()`** before the bash cells if you want only one Spark stack consuming RAM.

In [13]:
import os

# %%bash inherits the Jupyter kernel env
os.environ["FULLSTACK_REPO"] = str(REPO_ROOT.resolve())
os.environ["FULLSTACK_PARQUET_IN"] = str(OUTPUT_PARQUET.resolve())
os.environ["FULLSTACK_PARQUET_OUT"] = str(TOKENIZED_PARQUET.resolve())

### Build the JAR (once per Scala change)

Requires **`sbt`** on `PATH`. Equivalent to `cd scala/tokenize-email-job && sbt package`.


In [14]:
%%bash
set -euo pipefail
cd "${FULLSTACK_REPO}/scala/tokenize-email-job"
sbt package

[info] welcome to sbt 1.10.2 (Eclipse Adoptium Java 11.0.30)
[info] loading project definition from /Users/markcosciello/git/fullstack/scala/tokenize-email-job/project
[info] loading settings for project tokenize-email-job from build.sbt ...
[info] set current project to fullstack-tokenize-email-job (in build file:/Users/markcosciello/git/fullstack/scala/tokenize-email-job/)
[success] Total time: 8 s, completed May 1, 2026, 7:26:04 AM


### Run the tokenizer (`run-local.sh`)

Passes explicit input/output directories (overrides Scala defaults that hard-code `/Users/markcosciello/git/fullstack/...`).

In [15]:
%%bash
set -euo pipefail
cd "${FULLSTACK_REPO}"
./scala/tokenize-email-job/run-local.sh "${FULLSTACK_PARQUET_IN}" "${FULLSTACK_PARQUET_OUT}"

26/05/01 07:26:12 INFO SparkContext: Running Spark version 3.5.8
26/05/01 07:26:12 INFO SparkContext: OS info Mac OS X, 12.7.6, x86_64
26/05/01 07:26:12 INFO SparkContext: Java version 11.0.30
26/05/01 07:26:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/01 07:26:12 INFO ResourceUtils: ==============================================================
26/05/01 07:26:12 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/01 07:26:12 INFO ResourceUtils: ==============================================================
26/05/01 07:26:12 INFO SparkContext: Submitted application: tokenize-emails
26/05/01 07:26:12 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name:

+-----------+------+------------------+-------------------+-----------+------------+------------------+-------------------+--------------------+
|customer_id|name  |email             |ingested_at        |email_local|email_domain|email_local_tokens|email_domain_tokens|email_token_string  |
+-----------+------+------------------+-------------------+-----------+------------+------------------+-------------------+--------------------+
|3          |Carlos|carlos@example.com|2026-01-01 09:30:00|carlos     |example.com |[carlos]          |[example, com]     |carlos @ example.com|
|1          |Alice |alice@example.com |2026-01-01 09:30:00|alice      |example.com |[alice]           |[example, com]     |alice @ example.com |
|2          |Bob   |bob@example.com   |2026-01-01 09:30:00|bob        |example.com |[bob]             |[example, com]     |bob @ example.com   |
+-----------+------+------------------+-------------------+-----------+------------+------------------+-------------------+-------

26/05/01 07:26:24 INFO FileSourceStrategy: Pushed Filters: 
26/05/01 07:26:24 INFO FileSourceStrategy: Post-Scan Filters: 
26/05/01 07:26:24 INFO ParquetUtils: Using default output committer for Parquet: org.apache.parquet.hadoop.ParquetOutputCommitter
26/05/01 07:26:24 INFO FileOutputCommitter: File Output Committer Algorithm version is 1
26/05/01 07:26:24 INFO FileOutputCommitter: FileOutputCommitter skip cleanup _temporary folders under output directory:false, ignore cleanup failures: false
26/05/01 07:26:24 INFO SQLHadoopMapReduceCommitProtocol: Using user defined output committer class org.apache.parquet.hadoop.ParquetOutputCommitter
26/05/01 07:26:24 INFO FileOutputCommitter: File Output Committer Algorithm version is 1
26/05/01 07:26:24 INFO FileOutputCommitter: FileOutputCommitter skip cleanup _temporary folders under output directory:false, ignore cleanup failures: false
26/05/01 07:26:24 INFO SQLHadoopMapReduceCommitProtocol: Using output committer class org.apache.parquet.ha

### Read tokenized Parquet (PySpark)

Uses the notebook **`spark`** session to confirm the Scala job output.


In [16]:
tokenized_df = spark.read.parquet(str(TOKENIZED_PARQUET))
tokenized_df.orderBy("customer_id").show(truncate=False)
tokenized_df.printSchema()

+-----------+------+------------------+-------------------+-----------+------------+------------------+-------------------+--------------------+
|customer_id|name  |email             |ingested_at        |email_local|email_domain|email_local_tokens|email_domain_tokens|email_token_string  |
+-----------+------+------------------+-------------------+-----------+------------+------------------+-------------------+--------------------+
|1          |Alice |alice@example.com |2026-01-01 09:30:00|alice      |example.com |[alice]           |[example, com]     |alice @ example.com |
|2          |Bob   |bob@example.com   |2026-01-01 09:30:00|bob        |example.com |[bob]             |[example, com]     |bob @ example.com   |
|3          |Carlos|carlos@example.com|2026-01-01 09:30:00|carlos     |example.com |[carlos]          |[example, com]     |carlos @ example.com|
+-----------+------+------------------+-------------------+-----------+------------+------------------+-------------------+-------

## Optional: parity with packaged `customers_job` only

**Skip** if you already wrote Parquet and ran the Scala section—invoking **`run()`** overwrites **`output`** and stops Spark, so you would re-run the tokenizer afterward.

Use **`run()`** to match `poetry run python -m fullstack.jobs.customers_job` exactly.


In [17]:
from fullstack.jobs.customers_job import run
run()

26/05/01 07:27:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

+-----------+------+------------------+-------------------+
|customer_id|name  |email             |ingested_at        |
+-----------+------+------------------+-------------------+
|1          |Alice |alice@example.com |2026-01-01 09:30:00|
|2          |Bob   |bob@example.com   |2026-01-01 09:30:00|
|3          |Carlos|carlos@example.com|2026-01-01 09:30:00|
+-----------+------+------------------+-------------------+

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [18]:
spark.stop()  # uncomment when you're done